In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# ============================================
# КОНФИГУРАЦИЯ - МЕНЯЙТЕ ЭТИ ПАРАМЕТРЫ
# ============================================

# Размер выборки для обучения (None = все данные, число = сэмплирование)
TRAIN_SAMPLE_SIZE = 100000  # Поставьте None для всех данных или число для сэмпла
TEST_SAMPLE_SIZE = None      # Обычно тест не сэмплируем

# Параметры обучения
USE_LIGHT_MODELS = True      # True = быстрые модели, False = полные модели
CV_FOLDS = 3                 # Количество фолдов кросс-валидации (3 для скорости, 5 для качества)
THRESHOLD = 0.55             # Порог классификации (выше = меньше false positives)

# Какие модели использовать
USE_XGBOOST = True
USE_LIGHTGBM = True
USE_RANDOM_FOREST = False    # RF медленный на больших данных
USE_LOGISTIC = True

# TF-IDF параметры
MAX_TFIDF_FEATURES = 500     # Количество TF-IDF признаков (меньше = быстрее)

# Показывать ли прогресс
VERBOSE = True
SHOW_SCORES = True

# ============================================
# ИМПОРТ БИБЛИОТЕК
# ============================================

import re
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import fbeta_score, precision_score, recall_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix
import xgboost as xgb
import lightgbm as lgb

print("="*60)
print("DGA DOMAIN DETECTION - МАСШТАБИРУЕМОЕ РЕШЕНИЕ")
print("="*60)

# ============================================
# ЗАГРУЗКА ДАННЫХ
# ============================================

start_time = time.time()

print("\n[1/6] Загрузка данных...")
train_df = pd.read_csv('/kaggle/input/competitions/dga-domain-detection-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/dga-domain-detection-challenge/test.csv')

print(f"Исходный размер train: {train_df.shape[0]:,} записей")
print(f"Исходный размер test: {test_df.shape[0]:,} записей")

# Сэмплирование если нужно
if TRAIN_SAMPLE_SIZE and TRAIN_SAMPLE_SIZE < len(train_df):
    print(f"\nСэмплируем train до {TRAIN_SAMPLE_SIZE:,} записей...")
    # Сохраняем баланс классов при сэмплировании
    dga_samples = train_df[train_df['label'] == 1].sample(
        n=min(TRAIN_SAMPLE_SIZE//2, len(train_df[train_df['label'] == 1])), 
        random_state=42
    )
    legit_samples = train_df[train_df['label'] == 0].sample(
        n=min(TRAIN_SAMPLE_SIZE//2, len(train_df[train_df['label'] == 0])), 
        random_state=42
    )
    train_df = pd.concat([dga_samples, legit_samples]).sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"Размер после сэмплирования: {train_df.shape[0]:,} записей")

if TEST_SAMPLE_SIZE and TEST_SAMPLE_SIZE < len(test_df):
    print(f"Сэмплируем test до {TEST_SAMPLE_SIZE:,} записей...")
    test_df = test_df.sample(n=TEST_SAMPLE_SIZE, random_state=42)

print(f"Баланс классов в train:\n{train_df['label'].value_counts()}")
print(f"Баланс %: \n{train_df['label'].value_counts(normalize=True)}")

# ============================================
# ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ
# ============================================

print("\n[2/6] Извлечение признаков...")
feature_start = time.time()

def extract_features_batch(df, batch_name=""):
    """Быстрое извлечение признаков"""
    if VERBOSE:
        print(f"  Извлечение признаков {batch_name}...")
    
    features = pd.DataFrame()
    
    # Обработка NaN значений и приведение к строке
    df['domain'] = df['domain'].fillna('').astype(str)
    
    # Извлекаем SLD (second-level domain)
    df['sld'] = df['domain'].apply(lambda x: x.split('.')[0] if isinstance(x, str) and '.' in x else str(x))
    
    # Основные признаки (быстрые для вычисления)
    features['domain_length'] = df['domain'].str.len()
    features['sld_length'] = df['sld'].str.len()
    features['dot_count'] = df['domain'].str.count('\.')
    features['digit_count'] = df['sld'].str.count(r'\d')
    features['letter_count'] = df['sld'].str.count(r'[a-zA-Z]')
    
    # Производные признаки
    features['digit_ratio'] = features['digit_count'] / (features['sld_length'] + 1)
    features['letter_ratio'] = features['letter_count'] / (features['sld_length'] + 1)
    
    # Энтропия
    def fast_entropy(series):
        entropies = []
        for s in series:
            s = str(s) if not isinstance(s, str) else s
            if len(s) == 0:
                entropies.append(0)
                continue
            freq = {}
            for c in s:
                freq[c] = freq.get(c, 0) + 1
            entropy = 0
            for count in freq.values():
                prob = count / len(s)
                entropy -= prob * np.log2(prob)
            entropies.append(entropy)
        return entropies
    
    features['entropy'] = fast_entropy(df['sld'])
    
    # Дополнительные быстрые признаки
    features['unique_chars'] = df['sld'].apply(lambda x: len(set(str(x))))
    features['unique_ratio'] = features['unique_chars'] / (features['sld_length'] + 1)
    features['has_hyphen'] = df['domain'].str.contains('-', regex=False).astype(int)
    features['has_digit'] = df['sld'].str.contains(r'\d', regex=True).astype(int)
    
    # Количество гласных/согласных
    features['vowel_count'] = df['sld'].str.lower().str.count(r'[aeiou]')
    features['consonant_count'] = df['sld'].str.lower().str.count(r'[bcdfghjklmnpqrstvwxyz]')
    features['vowel_ratio'] = features['vowel_count'] / (features['consonant_count'] + 1)
    
    # Последовательные символы
    features['max_consec_digits'] = df['sld'].apply(
        lambda x: max([len(m) for m in re.findall(r'\d+', str(x))] + [0])
    )
    features['max_consec_consonants'] = df['sld'].apply(
        lambda x: max([len(m) for m in re.findall(r'[^aeiou\W\d]+', str(x).lower())] + [0])
    )
    
    return features

# Извлечение признаков
X_train_features = extract_features_batch(train_df, "train")
X_test_features = extract_features_batch(test_df, "test")

print(f"  Создано признаков: {X_train_features.shape[1]}")
print(f"  Время извлечения: {time.time() - feature_start:.1f} сек")

# ============================================
# TF-IDF ПРИЗНАКИ
# ============================================

print("\n[3/6] Создание TF-IDF признаков...")
tfidf_start = time.time()

tfidf_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(2, 3),
    max_features=MAX_TFIDF_FEATURES,
    lowercase=True,
    sublinear_tf=True  # Быстрее и часто лучше
)

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['sld'])
X_test_tfidf = tfidf_vectorizer.transform(test_df['sld'])

print(f"  TF-IDF признаков: {X_train_tfidf.shape[1]}")
print(f"  Время TF-IDF: {time.time() - tfidf_start:.1f} сек")

# ============================================
# ОБЪЕДИНЕНИЕ ПРИЗНАКОВ
# ============================================

print("\n[4/6] Объединение признаков...")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_features)
X_test_scaled = scaler.transform(X_test_features)

X_train_combined = hstack([csr_matrix(X_train_scaled), X_train_tfidf])
X_test_combined = hstack([csr_matrix(X_test_scaled), X_test_tfidf])

y = train_df['label'].values

print(f"  Финальный размер матрицы: {X_train_combined.shape}")
print(f"  Плотность матрицы: {(X_train_combined.nnz / (X_train_combined.shape[0] * X_train_combined.shape[1])) * 100:.2f}%")

# ============================================
# ОБУЧЕНИЕ МОДЕЛЕЙ
# ============================================

print("\n[5/6] Обучение моделей...")
model_start = time.time()

models = {}
model_scores = {}

# XGBoost
if USE_XGBOOST:
    print("  Обучение XGBoost...")
    if USE_LIGHT_MODELS:
        xgb_model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=4,
            learning_rate=0.1,
            scale_pos_weight=len(y[y==0]) / max(len(y[y==1]), 1),
            random_state=42,
            n_jobs=-1,
            use_label_encoder=False,
            eval_metric='logloss',
            verbosity=0
        )
    else:
        xgb_model = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            scale_pos_weight=len(y[y==0]) / max(len(y[y==1]), 1),
            random_state=42,
            n_jobs=-1,
            use_label_encoder=False,
            eval_metric='logloss',
            verbosity=0
        )
    models['XGBoost'] = xgb_model

# LightGBM
if USE_LIGHTGBM:
    print("  Обучение LightGBM...")
    if USE_LIGHT_MODELS:
        lgb_model = lgb.LGBMClassifier(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.1,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1,
            verbose=-1,
            force_col_wise=True  # Быстрее на больших данных
        )
    else:
        lgb_model = lgb.LGBMClassifier(
            n_estimators=300,
            max_depth=8,
            learning_rate=0.05,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1,
            verbose=-1,
            force_col_wise=True
        )
    models['LightGBM'] = lgb_model

# Random Forest (только если данных не слишком много)
if USE_RANDOM_FOREST and len(y) <= 200000:
    print("  Обучение Random Forest...")
    rf_model = RandomForestClassifier(
        n_estimators=100 if USE_LIGHT_MODELS else 200,
        max_depth=8,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    models['RandomForest'] = rf_model

# Logistic Regression
if USE_LOGISTIC:
    print("  Обучение Logistic Regression...")
    lr_model = LogisticRegression(
        C=0.1,
        class_weight='balanced',
        random_state=42,
        max_iter=1000,
        n_jobs=-1
    )
    models['LogisticRegression'] = lr_model

print(f"\n  Всего моделей для обучения: {len(models)}")

# Кросс-валидация
if SHOW_SCORES and CV_FOLDS > 0:
    print(f"\n  Кросс-валидация ({CV_FOLDS} фолдов)...")
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    
    for name, model in models.items():
        cv_scores_f05 = []
        cv_scores_precision = []
        cv_scores_recall = []
        
        print(f"    Оценка {name}...")
        for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_combined, y)):
            X_tr = X_train_combined[train_idx]
            X_val = X_train_combined[val_idx]
            y_tr = y[train_idx]
            y_val = y[val_idx]
            
            model.fit(X_tr, y_tr)
            y_pred_proba = model.predict_proba(X_val)[:, 1]
            y_pred = (y_pred_proba > THRESHOLD).astype(int)
            
            f05 = fbeta_score(y_val, y_pred, beta=0.5)
            precision = precision_score(y_val, y_pred)
            recall = recall_score(y_val, y_pred)
            
            cv_scores_f05.append(f05)
            cv_scores_precision.append(precision)
            cv_scores_recall.append(recall)
        
        model_scores[name] = {
            'F0.5': np.mean(cv_scores_f05),
            'F0.5_std': np.std(cv_scores_f05),
            'Precision': np.mean(cv_scores_precision),
            'Recall': np.mean(cv_scores_recall)
        }
        
        print(f"      F0.5: {model_scores[name]['F0.5']:.4f} (+/- {model_scores[name]['F0.5_std']:.4f})")
        print(f"      Precision: {model_scores[name]['Precision']:.4f}")
        print(f"      Recall: {model_scores[name]['Recall']:.4f}")

# Финальное обучение на всех данных
print(f"\n  Финальное обучение на всех данных...")
for name, model in models.items():
    print(f"    Обучение {name}...")
    model.fit(X_train_combined, y)

print(f"  Время обучения: {time.time() - model_start:.1f} сек")

# ============================================
# СОЗДАНИЕ ПРЕДСКАЗАНИЙ
# ============================================

print("\n[6/6] Создание предсказаний...")

# Ансамблирование
ensemble_pred = np.zeros(X_test_combined.shape[0])
weights = []

if USE_XGBOOST and USE_LIGHTGBM:
    weights = [0.4, 0.4, 0.2] if USE_LOGISTIC else [0.5, 0.5]
elif USE_XGBOOST:
    weights = [1.0]
elif USE_LIGHTGBM:
    weights = [1.0]

weight_sum = sum(weights)
for i, (name, model) in enumerate(models.items()):
    if i < len(weights):
        ensemble_pred += weights[i] * model.predict_proba(X_test_combined)[:, 1]

ensemble_pred /= weight_sum

# Применяем порог
final_predictions = (ensemble_pred > THRESHOLD).astype(int)

# Статистика предсказаний
dga_count = np.sum(final_predictions)
legit_count = len(final_predictions) - dga_count

print(f"\n  Распределение предсказаний:")
print(f"  DGA (1): {dga_count:,} ({dga_count/len(final_predictions)*100:.2f}%)")
print(f"  Legitimate (0): {legit_count:,} ({legit_count/len(final_predictions)*100:.2f}%)")

# ============================================
# СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# ============================================

submission = pd.DataFrame({
    'id': test_df.index,
    'label': final_predictions
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

# Сохраняем вероятности
probabilities_df = pd.DataFrame({
    'id': test_df.index,
    'probability': ensemble_pred
})
probabilities_df.to_csv('/kaggle/working/probabilities.csv', index=False)

# ============================================
# ИТОГОВЫЙ SCORE
# ============================================

total_time = time.time() - start_time

print("\n" + "="*60)
print("ИТОГОВЫЙ ОТЧЕТ")
print("="*60)
print(f"Конфигурация:")
print(f"  Размер train: {len(train_df):,} записей")
print(f"  Размер test: {len(test_df):,} записей")
print(f"  Признаков: {X_train_combined.shape[1]}")
print(f"  Моделей в ансамбле: {len(models)}")
print(f"  Порог классификации: {THRESHOLD}")
print(f"  Folds CV: {CV_FOLDS}")

if SHOW_SCORES and model_scores:
    print(f"\nSCORES (Cross-Validation):")
    print("-" * 40)
    for name, scores in model_scores.items():
        print(f"\n  {name}:")
        print(f"    F0.5 Score: {scores['F0.5']:.4f} (±{scores['F0.5_std']:.4f})")
        print(f"    Precision:  {scores['Precision']:.4f}")
        print(f"    Recall:     {scores['Recall']:.4f}")
    
    # Средний скор по ансамблю
    avg_f05 = np.mean([s['F0.5'] for s in model_scores.values()])
    avg_precision = np.mean([s['Precision'] for s in model_scores.values()])
    avg_recall = np.mean([s['Recall'] for s in model_scores.values()])
    
    print(f"\n  СРЕДНЕЕ ПО АНСАМБЛЮ:")
    print(f"    F0.5 Score: {avg_f05:.4f}")
    print(f"    Precision:  {avg_precision:.4f}")
    print(f"    Recall:     {avg_recall:.4f}")

print(f"\nПроизводительность:")
print(f"  Общее время: {total_time:.1f} сек ({total_time/60:.1f} мин)")
print(f"  Скорость: {len(train_df)/total_time:.0f} записей/сек")

print(f"\nФайлы сохранены:")
print(f"  /kaggle/working/submission.csv - предсказания")
print(f"  /kaggle/working/probabilities.csv - вероятности")

print("\n" + "="*60)
print("ГОТОВО! Можно отправлять submission.csv")
print("="*60)

<>:132: SyntaxWarning: invalid escape sequence '\.'
<>:132: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_16/2507984794.py:132: SyntaxWarning: invalid escape sequence '\.'
  features['dot_count'] = df['domain'].str.count('\.')


/kaggle/input/competitions/dga-domain-detection-challenge/sample_submission.csv
/kaggle/input/competitions/dga-domain-detection-challenge/train.csv
/kaggle/input/competitions/dga-domain-detection-challenge/test.csv
DGA DOMAIN DETECTION - МАСШТАБИРУЕМОЕ РЕШЕНИЕ

[1/6] Загрузка данных...
Исходный размер train: 17,719,790 записей
Исходный размер test: 7,594,197 записей

Сэмплируем train до 100,000 записей...
Размер после сэмплирования: 100,000 записей
Баланс классов в train:
label
0    50000
1    50000
Name: count, dtype: int64
Баланс %: 
label
0    0.5
1    0.5
Name: proportion, dtype: float64

[2/6] Извлечение признаков...
  Извлечение признаков train...
  Извлечение признаков test...
  Создано признаков: 17
  Время извлечения: 220.3 сек

[3/6] Создание TF-IDF признаков...
  TF-IDF признаков: 500
  Время TF-IDF: 106.3 сек

[4/6] Объединение признаков...
  Финальный размер матрицы: (100000, 517)
  Плотность матрицы: 5.19%

[5/6] Обучение моделей...
  Обучение XGBoost...
  Обучение LightG